<a href="https://colab.research.google.com/github/SkyWarLimit/PemMes_17/blob/main/JS09/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faiss-cpu hnswlib annoy

import numpy as np
import pandas as pd
import time
import faiss
import hnswlib
from annoy import AnnoyIndex

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.5 MB/s  0:00:00
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=ade4f089027fa1fb48f8dc3c591c2d9e8a38fe4d4bb3422915231018f7767daa
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=b59965f2b77ab9530c834b4bf231bedf03494941733faa4d917ba867657cfe6e
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415

In [3]:
def benchmark_annoy(X, query, n_neighbors, metric='euclidean'):
    d = X.shape[1]
    # ANNOY mendukung 'euclidean', 'angular', 'manhattan', 'hamming'
    t = AnnoyIndex(d, metric)
    for i in range(X.shape[0]):
        t.add_item(i, X[i])
    t.build(10) # 10 trees
    start_time = time.time()
    t.get_nns_by_vector(query, n_neighbors)
    return time.time() - start_time

def benchmark_faiss(X, query, n_neighbors, metric='l2'):
    d = X.shape[1]
    # FAISS menggunakan IndexFlatL2 untuk Euclidean dan IndexFlatIP untuk Inner Product
    if metric == 'l2':
        index = faiss.IndexFlatL2(d)
    else:
        index = faiss.IndexFlatIP(d)
    index.add(X)
    start_time = time.time()
    index.search(np.array([query]), n_neighbors)
    return time.time() - start_time

def benchmark_hnsw(X, query, n_neighbors, metric='l2'):
    d = X.shape[1]
    # HNSWLIB mendukung 'l2', 'ip', 'cosine'
    p = hnswlib.Index(space=metric, dim=d)
    p.init_index(max_elements=X.shape[0], ef_construction=200, M=16)
    p.add_items(X)
    p.set_ef(50)
    start_time = time.time()
    p.knn_query(np.array([query]), k=n_neighbors)
    return time.time() - start_time

In [4]:
# Konfigurasi: (Jumlah Data, Dimensi, Metrik Jarak)
configs = [
    (1000, 2, 'euclidean'),
    (1000, 2, 'cosine'),
    (1000000, 5, 'euclidean'),
    (1000000, 5, 'cosine')
]

results = []
n_neighbors = 10

print("Memulai proses komputasi... (Mungkin memakan waktu beberapa saat untuk data 1 juta)")

for n, d, metric in configs:
    # Generate data acak (dummy) bertipe float32
    X = np.random.random((n, d)).astype('float32')
    query = X[0] # Menggunakan data pertama sebagai query pencarian

    # Penyesuaian nama metrik untuk masing-masing library
    annoy_m = 'euclidean' if metric == 'euclidean' else 'angular'
    faiss_m = 'l2' if metric == 'euclidean' else 'ip'  # FAISS ip (inner product) ~ cosine jika dinormalisasi
    hnsw_m = 'l2' if metric == 'euclidean' else 'cosine'

    # Uji waktu masing-masing
    t_annoy = benchmark_annoy(X, query, n_neighbors, annoy_m)
    t_faiss = benchmark_faiss(X, query, n_neighbors, faiss_m)
    t_hnsw = benchmark_hnsw(X, query, n_neighbors, hnsw_m)

    results.append({
        'Jumlah Data/Dimensi': f"{n}/{d}D ({metric})",
        'ANNOY': f"{t_annoy:.6f}",
        'FAISS': f"{t_faiss:.6f}",
        'HNSW': f"{t_hnsw:.6f}"
    })

# Menampilkan hasil komparasi menjadi Dataframe seperti tabel pada soal
df_results = pd.DataFrame(results)
display(df_results)

Memulai proses komputasi... (Mungkin memakan waktu beberapa saat untuk data 1 juta)


,Jumlah Data/Dimensi,ANNOY,FAISS,HNSW
0,1000/2D (euclidean),0.000064,0.000129,0.000165
1,1000/2D (cosine),0.000046,0.000060,0.000076
2,1000000/5D (euclidean),0.000119,0.022455,0.000181
3,1000000/5D (cosine),0.000143,0.012104,0.000138
